# llm_figures

> Phase 10 figures: welfare vs team size per channel and task, harness benefit by model size, tokens as compute and the equal-budget arm, numpy versus LLM cell means, and the arms.

In [ ]:
#| default_exp llm_figures

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from explRL.llm_report import _f, _sel

COLORS = dict(none='tab:gray', blackboard='tab:blue', messages='tab:orange', hierarchical='tab:green')


def _legend(ax):
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=8)


def _save(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=110, metadata={'Software': None})
    plt.close(fig)
    return path


def figWelfare(S, path):
    "Welfare per agent-turn vs team size, one line per channel, one panel per task (main grid, 95% CIs)."
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    for ax, task in zip(axs, ('forage', 'board')):
        for ch, col in COLORS.items():
            pts = sorted((N, s) for (t, c, N), s in S['main'].items() if t == task and c == ch)
            if pts:
                N = [p[0] for p in pts]
                m = np.array([p[1]['mean'] for p in pts])
                lo, hi = np.array([p[1]['ci'][0] for p in pts]), np.array([p[1]['ci'][1] for p in pts])
                ax.errorbar(N, m, yerr=[m - lo, hi - m], label=ch, color=col, marker='o', capsize=3)
        ax.set(xscale='log', xlabel='team size N', ylabel='welfare per agent-turn', title=f'{task} (Claude Haiku 5.5)')
        ax.set_xticks([1, 2, 4, 8], ['1', '2', '4', '8'])
        ax.minorticks_off()
        ax.axhline(0, color='k', lw=0.5)
    _legend(axs[0])
    return _save(fig, path)


def figModel(S, path):
    "Harness benefit on the task board, Haiku vs Sonnet, sizes 1 and 4."
    fig, ax = plt.subplots(figsize=(7, 4))
    keys = [(ch, N) for ch in ('blackboard', 'messages', 'hierarchical') for N in (1, 4)]
    x = np.arange(len(keys))
    for off, (lab, k) in zip((-0.2, 0.2), (('Haiku 5.5', 'benefit'), ('Sonnet 5.5', 'sonnetBenefit'))):
        st = [S[k].get(('board', ch, N), dict(mean=np.nan, ci=[np.nan, np.nan])) for ch, N in keys]
        m = np.array([s['mean'] for s in st])
        err = [m - np.array([s['ci'][0] for s in st]), np.array([s['ci'][1] for s in st]) - m]
        ax.bar(x + off, m, 0.4, yerr=err, label=lab, capsize=3)
    ax.set_xticks(x, [f'{ch}\nN={N}' for ch, N in keys], fontsize=8)
    ax.axhline(0, color='k', lw=0.5)
    ax.set(ylabel='welfare gain over no channel', title='Does a channel help less with a stronger model? (task board)')
    _legend(ax)
    return _save(fig, path)


def figTokens(rows, S, path):
    "Team welfare vs total tokens per run (main grid), and the equal-budget arm."
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    for r in _sel(rows, study='main', runner='llm'):
        tok = sum(_f(r[k]) for k in ('inputTokens', 'cacheWriteTokens', 'cacheReadTokens', 'outputTokens'))
        team = _f(r['welfare']) * _f(r['N']) * _f(r['turns'])
        axs[0].scatter(tok, team, color=COLORS[r['channel']], marker='o' if r['task'] == 'forage' else 's', s=14)
    axs[0].set(xscale='log', xlabel='total tokens per run', ylabel='team welfare',
               title='Tokens as compute (circles: foraging, squares: board)')
    axs[0].xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
    for ch in ('none', 'blackboard'):
        pts = sorted((N, s) for (c, N), s in S['budget'].items() if c == ch)
        if pts:
            m = np.array([p[1]['mean'] for p in pts])
            err = [m - np.array([p[1]['ci'][0] for p in pts]), np.array([p[1]['ci'][1] for p in pts]) - m]
            axs[1].errorbar([p[0] for p in pts], m, yerr=err, color=COLORS[ch], marker='o', capsize=3, label=ch)
    axs[1].set(xscale='log', xlabel='team size N (200 agent-turns in all)', ylabel='welfare per agent-turn',
               title='Equal budget (foraging)')
    axs[1].set_xticks([1, 2, 4, 8], ['1', '2', '4', '8'])
    axs[1].minorticks_off()
    _legend(axs[1])
    return _save(fig, path)


def figNumpy(rows, numpyRows, S, path):
    "Cell means of welfare: LLM vs numpy, per task, with Spearman's rho."
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    for ax, task in zip(axs, ('forage', 'board')):
        for ch, col in COLORS.items():
            for N in (1, 2, 4, 8):
                a = [_f(r['welfare']) for r in _sel(rows, study='main', runner='llm', task=task, channel=ch, N=N)]
                b = [_f(r['welfare']) for r in _sel(list(numpyRows), task=task, channel=ch, N=N)]
                if a and b:
                    ax.scatter(np.mean(b), np.mean(a), color=col, s=10 + 8 * N, label=ch if N == 1 else None)
        v = S['numpy'].get(task)
        rho = f"rho {v['rho']:.2f} [{v['ci'][0]:.2f}, {v['ci'][1]:.2f}]" if v else ''
        ax.set(xlabel='numpy welfare per agent-turn', ylabel='LLM welfare per agent-turn', title=f'{task}: {rho}')
    _legend(axs[0])
    return _save(fig, path)


def figArms(S, path):
    "Each arm's difference from its control (95% CI)."
    items = list(S['arms'].items())
    fig, ax = plt.subplots(figsize=(10, 0.35 * len(items) + 1.2))
    y = np.arange(len(items))
    m = np.array([s['mean'] for _, s in items])
    err = [m - np.array([s['ci'][0] for _, s in items]), np.array([s['ci'][1] for _, s in items]) - m]
    ax.errorbar(m, y, xerr=err, fmt='o', capsize=3)
    ax.set_yticks(y, [k for k, _ in items], fontsize=7)
    ax.axvline(0, color='k', lw=0.5)
    ax.invert_yaxis()
    ax.set(xlabel='difference from control', title='Prompt-only arms and the liar')
    return _save(fig, path)


def figTeamTotal(rows, path, study='main', periods=dict(forage=40.0, board=10.0)):
    "Cumulative team harvest over world time (from the teamRate columns), one line per channel, panels by task and N."
    tasks = [t for t in ('forage', 'board') if _sel(rows, study=study, task=t)]
    Ns = sorted({int(_f(r['N'])) for r in _sel(rows, study=study)})
    if not tasks:
        fig, ax = plt.subplots(figsize=(4, 2))
        ax.text(0.5, 0.5, 'no rows to plot', ha='center', va='center')
        ax.set_axis_off()
        return _save(fig, path)
    fig, axs = plt.subplots(len(tasks), len(Ns), figsize=(3 * len(Ns), 2.6 * len(tasks)), squeeze=False)
    for a, task in enumerate(tasks):
        for b, N in enumerate(Ns):
            ax, p = axs[a][b], periods[task]
            for ch, col in COLORS.items():
                rs = _sel(rows, study=study, task=task, channel=ch, N=N)
                K = min((sum(k.startswith('teamRate_') and _f(r[k]) == _f(r[k]) for k in r) for r in rs), default=0)
                if K:
                    m = np.mean([[_f(r[f'teamRate_{k}']) * p for k in range(K)] for r in rs], 0).cumsum()
                    ax.plot(p * np.arange(1, K + 1), m, color=col, marker='.', label=ch)
            ax.set_title(f'{task}, N = {N}', fontsize=9)
            ax.set_xlabel('world time (s)' if task == 'forage' else 'turns', fontsize=8)
        axs[a][0].set_ylabel('team harvest, cumulative', fontsize=8)
    _legend(axs[0][0])
    return _save(fig, path)


def writeFigures(out, rows, numpyRows, S):
    return [figWelfare(S, out / 'welfare.png'), figModel(S, out / 'model_size.png'), figTokens(rows, S, out / 'tokens.png'),
            figNumpy(rows, numpyRows, S, out / 'numpy_vs_llm.png'), figArms(S, out / 'arms.png'),
            figTeamTotal(rows, out / 'team_total.png')]

## Tests

In [ ]:
import tempfile
from pathlib import Path
from explRL.llm_figures import *
from explRL.llm_report import summaries
from explRL.bench_grid import runGrid
from explRL.llm_grid import quickGrid, runAnyCell, withClient

rows = runGrid(withClient(quickGrid(), dict(kind='fake')), workers=1, run=runAnyCell)
llm, npy = [r for r in rows if r['runner'] == 'llm'], [r for r in rows if r['runner'] == 'numpy']
out = Path(tempfile.mkdtemp())
paths = writeFigures(out, llm, npy, summaries(llm, npy))
assert [p.name for p in paths] == ['welfare.png', 'model_size.png', 'tokens.png', 'numpy_vs_llm.png', 'arms.png', 'team_total.png']
assert all(p.stat().st_size > 10000 for p in paths)
assert figTeamTotal([], out / 'empty.png').stat().st_size > 0

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()